In [236]:
# Loading MNIST dataset
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import platform
import torch.nn.functional as F

# Apple Silicon / macOS: use Metal (MPS), else CPU.
_is_mac = platform.system() == "Darwin"
if _is_mac and torch.backends.mps.is_available():
    device = torch.device("mps")

else:
    device = torch.device("cpu")
print("Using device:", device)

# Load the MNIST dataset
mnist_train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
X_train = mnist_train_data.data.to(device=device, dtype=torch.float32)
X_train = (X_train - X_train.mean()) / X_train.std()
y_train = mnist_train_data.targets.to(device)


mnist_test_data = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
X_test = mnist_test_data.data.to(device=device, dtype=torch.float32)
X_test = (X_test - X_test.mean()) / X_test.std()
y_test = mnist_test_data.targets.to(device)


# Compute the score of the model (vectorized; avoid `if tensor == tensor` in a Python loop)
def score(ys_pred, ys):
    return (ys_pred == ys).float().mean().item()


Using device: mps


In [241]:
# Creating the CNN model
class CNN:
    def __init__(self):
        self.filter_size_1 = 3
        self.filter_size_2 = 2
        self.filter_size_3 = 2
        self.num_filters_1 = 32
        self.num_filters_2 = 16
        self.padding = 0
        self.stride = 1

        device, dtype = torch.device("mps"), torch.float32
        self.h_in, self.w_in = int(28), int(28)
        h_conv_1 = (self.h_in + 2 * self.padding - self.filter_size_1) // self.stride + 1
        w_conv_1 = (self.w_in + 2 * self.padding - self.filter_size_1) // self.stride + 1
        h_pool_1 = h_conv_1 // 2 
        w_pool_1 = w_conv_1 // 2 

        h_conv_2 = (h_pool_1 + 2 * self.padding - self.filter_size_2) // self.stride + 1
        w_conv_2 = (w_pool_1 + 2 * self.padding - self.filter_size_2) // self.stride + 1
        h_pool_2 = h_conv_2 // 2 
        w_pool_2 = w_conv_2 // 2 
        
        flat_dim = self.num_filters_2 * h_pool_2 * w_pool_2

        # Convolutional layer 1
        self.W1 = (
            torch.randn(
                self.num_filters_1,
                1,
                self.filter_size_1,
                self.filter_size_1,
                device=device,
                dtype=dtype,
            )
            * 0.05
        ).detach().requires_grad_()
        self.b1 = torch.zeros(self.num_filters_1, device=device, dtype=dtype, requires_grad=True)

        # Convolutional layer 2 
        self.W2 = (
            torch.randn(
                self.num_filters_2,
                self.num_filters_1,
                self.filter_size_2,
                self.filter_size_2,
                device=device,
                dtype=dtype,
            )
            * 0.05
        ).detach().requires_grad_()
        self.b2 = torch.zeros(self.num_filters_2, device=device, dtype=dtype, requires_grad=True)

        # Linear layer 1
        self.hidden = 64
        self.W3 = (
            torch.randn(flat_dim, self.hidden, device=device, dtype=dtype) * 0.05
        ).detach().requires_grad_()
        self.b3 = torch.zeros(self.hidden, device=device, dtype=dtype, requires_grad=True)

        # Linear layer 2
        self.W4 = (
            torch.randn(self.hidden, 10, device=device, dtype=dtype) * 0.05
        ).detach().requires_grad_()
        self.b4 = torch.zeros(10, device=device, dtype=dtype, requires_grad=True)

        self.params = [self.W1, self.b1, self.W2, self.b2, self.W3, self.b3, self.W4, self.b4]

    def train(self, lr):
        """Plain SGD. Use lr_decay < 1 (e.g. 0.95) to multiply the LR after each epoch."""
        batch_size = 1024
        num_epochs = 5
        n = X_train.shape[0]
        current_lr = float(lr)
        for _ in range(num_epochs):
            shuffled_indices = torch.randperm(n, device=X_train.device)
            for j in range(0, n, batch_size):
                Xs = X_train[shuffled_indices[j : j + batch_size]]
                ys = y_train[shuffled_indices[j : j + batch_size]]

                # Forward pass
                Z = F.conv2d(
                    Xs.unsqueeze(1),
                    self.W1,
                    self.b1,
                    stride=self.stride,
                    padding=self.padding,
                )
                # ReLU activation
                Z = F.relu(Z)
                # Max-pooling layer
                Z = F.max_pool2d(Z, kernel_size=2, stride=2)
                # Convolutional layer 2 (Z is NCHW after conv1+pool — use conv2d)
                Z = F.conv2d(
                    Z,
                    self.W2,
                    self.b2,
                    stride=self.stride,
                    padding=self.padding,
                )
                # ReLU activation
                Z = F.relu(Z)
                # Max-pooling layer
                Z = F.max_pool2d(Z, kernel_size=2, stride=2)
                # Flatten the output
                Z = Z.view(Z.shape[0], -1)
                # Linear layer
                Z = Z @ self.W3 + self.b3
                # ReLU activation
                Z = F.relu(Z)
                # Linear layer 2
                Z = Z @ self.W4 + self.b4
                loss = F.cross_entropy(Z, ys.long())


                # Backward pass
                for p in self.params:
                    p.grad = None
                loss.backward()
                with torch.no_grad():
                    for param in self.params:
                        if param.grad is not None:
                            param -= current_lr * param.grad


        print("The loss is: ", loss.detach().item())

    def predict(self, X):
        with torch.no_grad():
             # Forward pass
            Z = F.conv2d(
                X.unsqueeze(1),
                self.W1,
                self.b1,
                stride=self.stride,
                padding=self.padding,
            )
            # ReLU activation
            Z = F.relu(Z)
            # Max-pooling layer
            Z = F.max_pool2d(Z, kernel_size=2, stride=2)
            # Convolutional layer 2
            Z = F.conv2d(
                Z,
                self.W2,
                self.b2,
                stride=self.stride,
                padding=self.padding,
            )
            # ReLU activation   
            Z = F.relu(Z)
            # Max-pooling layer
            Z = F.max_pool2d(Z, kernel_size=2, stride=2)
            # Flatten the output
            Z = Z.view(Z.shape[0], -1)
            # Linear layer
            Z = Z @ self.W3 + self.b3
            # ReLU activation
            Z = F.relu(Z)
            # Linear layer 2
            Z = Z @ self.W4 + self.b4
            
            return Z.argmax(dim=1)

In [242]:
def compute_accuracy(CNN):
    with torch.no_grad():
        train_acc = score(CNN.predict(X_train), y_train)
        test_acc = score(CNN.predict(X_test), y_test)
    print(f"Train accuracy (same data as loss): {100 * train_acc:.2f}%")
    print(f"Test accuracy: {100 * test_acc:.2f}%")


In [243]:
SGD = CNN()

In [245]:
SGD.train(0.1)

The loss is:  0.10885825753211975


In [250]:
SGD.train(0.001)


The loss is:  0.06890460848808289


In [251]:
compute_accuracy(SGD)

Train accuracy (same data as loss): 97.61%
Test accuracy: 97.76%
